# Task 3: RAG chatbot using results from task 1

This notebook builds a RAG pipeline using langchain, embeddings, and a vector store, but it's specifically tailored to the 4-file CSV structure.

The Pipeline:

1. Load Data

2. Pre-process Data

3. Create Documents

4. Index

5. RAG Chain

In [ ]:
!pip install -U faiss-cpu==1.9.0 langchain==0.3.7    langchain-core==0.3.15    langchain-community==0.3.3    langchain-huggingface==0.1.2    langchain-google-genai==2.0.5    sentence-transformers==3.0.1    pydantic==2.9.2    accelerate==0.34.2

In [ ]:
import pandas as pd
import ast
import io
import os
import re
import time

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_google_genai import ChatGoogleGenerativeAI

import google.generativeai as genai
from google.colab import userdata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


 # Chatbot

In [ ]:
disease_desc_df = pd.read_csv('disease_desc.csv').set_index('Disease')
symptom_desc_df = pd.read_csv('symptoms_desc.csv').set_index('Symptom')
precaution_df = pd.read_csv('precaution.csv').set_index('Disease')
itemsets_df = pd.read_csv('itemsets.csv')

Load and Process All Files  

This is where we perform the "unpivot" and merge all data sources.

In [ ]:
# Helper function
def parse_symptom_string(s):
    if not isinstance(s, str):
        return []

    s_cleaned = s.strip('[] ')

    if not s_cleaned:
        return []

    def is_number(item_str):
        try:
            float(item_str)
            return True
        except ValueError:
            return False

    all_items = [item.strip() for item in s_cleaned.split(',')]
    symptoms_only = [item for item in all_items if item and not is_number(item)]

    return symptoms_only

In [ ]:
# 1. Process itemsets_df
rule_columns = itemsets_df.columns.drop('Disease')

itemsets_long = itemsets_df.melt(
    id_vars=['Disease'],
    value_vars = rule_columns,
    var_name='Rule_Name',
    value_name='symptom_set'
)

itemsets_long = itemsets_long.dropna(subset=['symptom_set'])
itemsets_long = itemsets_long[['Disease', 'symptom_set']]

itemsets_long['symptom_set'] = itemsets_long['symptom_set'].apply(parse_symptom_string)

# 2. Create a master symptom list for each disease
all_disease_symptoms = {}
for disease, group in itemsets_long.groupby('Disease'):
    unique_symptoms = set().union(*group['symptom_set'])
    all_disease_symptoms[disease] = unique_symptoms

print("--- Master Symptom Map (from itemsets.csv) ---")
print(all_disease_symptoms)

# 3. Build the final LangChain Documents
langchain_documents = []

for disease, row in disease_desc_df.iterrows():
    description = row['Description']

    symptoms = all_disease_symptoms.get(disease, set())
    symptom_list_formatted = []
    for symptom in symptoms:
        s_desc = symptom_desc_df.loc[symptom]['Description'] if symptom in symptom_desc_df.index else "No description."
        symptom_list_formatted.append(f"  - {symptom.capitalize()}: {s_desc}")
    symptoms_str = "\n".join(symptom_list_formatted)

    try:
        prec_row = precaution_df.loc[disease]
        precautions = [p for p in prec_row if pd.notna(p)]
        precautions_str = "\n".join(f"  - {p}" for p in precautions)
    except KeyError:
        precautions_str = "  - No precautions listed."

    page_content = f"""
      Disease: {disease}
      Description: {description}

      Common Symptoms (derived from frequent itemsets):
      {symptoms_str if symptoms else "  - No specific symptoms listed in itemsets."}

      Precautions:
      {precautions_str}
      """

    doc = Document(
        page_content=page_content,
        metadata={
            "disease": disease,
            "source": "medical_db_v2"
        }
    )
    langchain_documents.append(doc)

--- Master Symptom Map (from itemsets.csv) ---
{'(vertigo) Paroymsal  Positional Vertigo': {'nausea', 'loss_of_balance', 'spinning_movements', 'unsteadiness', 'headache', 'vomiting'}, 'AIDS': {'patches_in_throat', 'muscle_wasting', 'high_fever', 'extra_marital_contacts'}, 'Acne': {'scarring', 'blackheads', 'skin_rash', 'pus_filled_pimples'}, 'Alcoholic hepatitis': {'history_of_alcohol_consumption', 'swelling_of_stomach', 'abdominal_pain', 'distention_of_abdomen', 'yellowish_skin', 'fluid_overload', 'vomiting'}, 'Allergy': {'continuous_sneezing', 'watering_from_eyes', 'shivering', 'chills'}, 'Arthritis': {'swelling_joints', 'movement_stiffness', 'stiff_neck', 'muscle_weakness', 'painful_walking'}, 'Bronchial Asthma': {'mucoid_sputum', 'fatigue', 'high_fever', 'breathlessness', 'cough', 'family_history'}, 'Cervical spondylosis': {'loss_of_balance', 'dizziness', 'neck_pain', 'back_pain', 'weakness_in_limbs'}, 'Chicken pox': {'mild_fever', 'swelled_lymph_nodes', 'fatigue', 'itching', 'leth

In [ ]:
# Check the first document
print("\n--- Example LangChain Document ---")
print(langchain_documents[1].page_content)
print(f"Metadata: {langchain_documents[1].metadata}")


--- Example LangChain Document ---

Disease: AIDS
Description: Acquired immunodeficiency syndrome (AIDS) is a chronic, potentially life-threatening condition caused by the human immunodeficiency virus (HIV). By damaging your immune system, HIV interferes with your body's ability to fight infection and disease.

Common Symptoms (derived from frequent itemsets):
  - Patches_in_throat: Spots, streaks or areas of discoloration (often white or yellow) on the throat or tonsils, commonly caused by viral, bacterial or fungal infections.
  - Muscle_wasting: the process of losing muscle mass, which leads to decreased muscle strength and function
  - High_fever: A body temperature significantly above normal (e.g., above ~38?°C for adults) resulting from the bodys immune response to infection or illness.
  - Extra_marital_contacts: intimate sexual contacts outside of a committed (married) relationship

Precautions:
  - avoid open cuts
  - wear ppe if possible
  - consult doctor
  - follow up

Me

Create the Vector Store

In [ ]:
# 1. Initialize the embedding model
print("Loading local embedding model (all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Create the vector store from our documents
print("\nCreating vector store...")
vector_store = FAISS.from_documents(langchain_documents, embeddings)

# 3. Create the retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Test the retriever
query = "I have a high temperature and my whole body hurts"
retrieved_docs = retriever.get_relevant_documents(query)

print(f"\n--- Test Retrieval for: '{query}' ---")
for doc in retrieved_docs:
    print(f"Retrieved Disease: {doc.metadata['disease']}")

Loading local embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Creating vector store...

--- Test Retrieval for: 'I have a high temperature and my whole body hurts' ---
Retrieved Disease: Allergy
Retrieved Disease: Typhoid
Retrieved Disease: Common Cold


/tmp/ipython-input-1912771516.py:16: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)


Create the LangChain RAG Chain

In [ ]:
# 1. Initialize the Chat Model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                             temperature=0.3,
                             google_api_key=userdata.get('GOOGLE_API_KEY'))

# 2. Define the Prompt Template
prompt = ChatPromptTemplate.from_template("""
You are an empathetic and knowledgeable medical assistant chatbot.

Your goal is to help users understand possible conditions using **only the information in the provided context**.
Do not invent facts, symptoms, or advice that aren’t mentioned in the context.

---

### 🧩 What to Do
1. Read the user query ({input}) and the retrieved context ({context}).
2. Identify the **single most relevant disease** that fits the query.
   - If the user describes **symptoms**, relate them naturally to that disease.
   - If the user asks a **general question**, just explain the disease directly.
3. Share the “Description” and “Precautions” from the matched disease — in smooth, natural language.
4. End with a gentle reminder that this isn’t medical advice.

---

### 🩺 Response Style
- Write in a calm, caring tone (as if talking to a friend).
- Avoid robotic phrases like *“Based on your symptoms…”*.
- Vary sentence structure naturally.
- Combine short and medium-length sentences for rhythm.
- Always mention the disease name clearly and use the information from context only.

---

### 🗒️ Example Template (don’t copy literally — use as guidance)

🩺 **User Query:** {input}

🤖 **Assistant Response:**
If the query describes symptoms:
> I’m sorry you’re feeling under the weather. From what you’ve described, it seems quite similar to **{{disease name}}**.
> {{Brief, natural paraphrase of the description from context.}}

If it’s a general question:
> Sure! Here’s what the information says about **{{disease name}}**.
> {{Natural summary of description.}}

Then continue with:
> To take care of yourself, the information suggests:
> - {{Precaution 1}}
> - {{Precaution 2}}
> - {{Precaution 3}}

> Remember, this is general information from the provided source — it’s always best to check with a healthcare professional for proper diagnosis and care.
""")

# 3. Create the RAG chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("LangChain RAG chain created successfully.")

LangChain RAG chain created successfully.


Putting It All Together: The Final Chatbot

In [ ]:
def ask_chatbot(query: str):
    """
    Asks the RAG chain a question.
    """
    print(f"🩺 User Query: {query}")

    response = rag_chain.invoke({"input": query})

    print("\n🤖 Assistant Response:")
    print(response['answer'])

    print("\n--- (Debug) Retrieved Documents: ---")
    for doc in response['context']:
        print(f"  - Matched Disease: {doc.metadata['disease']}")

# 1. Classic symptom-based query (should match “Common Cold”)
ask_chatbot("I have a high temperature, cough, and my whole body aches. What should I do?")

print("\n" + "="*50 + "\n")

# 2. Ambiguous symptom description (model should pick best match)
ask_chatbot("I'm feeling tired and have a mild fever.")

print("\n" + "="*50 + "\n")

# 3. Direct informational query (should match “Pneumonia”)
ask_chatbot("What are the precautions for pneumonia?")

print("\n" + "="*50 + "\n")

# 4. User describes symptoms and suspects a disease
ask_chatbot("I'm coughing a lot and have chest pain — could this be bronchial asthma?")

print("\n" + "="*50 + "\n")

# 5. User asks about symptoms directly
ask_chatbot("What are the symptoms of typhoid?")

print("\n" + "="*50 + "\n")

# 6. Query about difference between diseases (should pick one best match)
ask_chatbot("How is dengue different from malaria?")

print("\n" + "="*50 + "\n")

# 7. No match scenario (for testing fallback response)
ask_chatbot("My Wi-Fi isn't working properly.")


🩺 User Query: I have a high temperature, cough, and my whole body aches. What should I do?

🤖 Assistant Response:
I’m sorry you’re feeling unwell. From what you've described – a high temperature, cough, and whole body aches – it sounds quite similar to symptoms associated with a **Common Cold**.

The common cold is a viral infection that affects your nose and throat, which is part of your upper respiratory tract. While it can certainly make you feel miserable, it's generally considered harmless. Many different types of viruses can be responsible for causing a common cold.

To help take care of yourself, the information suggests:
*   Drinking vitamin C rich drinks
*   Taking vapour
*   Avoiding cold food
*   Keeping your fever in check

Remember, this is general information from the provided source — it’s always best to check with a healthcare professional for proper diagnosis and care.

--- (Debug) Retrieved Documents: ---
  - Matched Disease: Allergy
  - Matched Disease: Common Cold
 